# 06_Face YOLO Experiment

## Objective
The objective of this notebook is to experiment with YOLO-based face classification/detection settings and analyze the results.

## Method
YOLO experiment workflow for training/evaluation and result analysis.

## Input
Face images and YOLO experiment configuration.

## Processing Tasks

1. (To be completed based on notebook section headings.)

## Output
- YOLO experiment metrics.
- Experiment visual artifacts.
- Recommended checkpoint for downstream usage.

## Purpose
Assess YOLO performance and choose the best setup for integration.


In [ ]:
from pathlib import Path
from PIL import Image
from collections import Counter

PROJECT_ROOT = Path(r"E:\Multimodal Biometric Verification System")

FACE_CROPPED_PATH = (
    PROJECT_ROOT
    / "Dataset"
    / "Processed"
    / "Face"
    / "Face_Cropped"
)

users = ["User_01", "User_02"]

for user in users:

    user_path = FACE_CROPPED_PATH / user

    dimensions = Counter()
    modes = Counter()

    for image_path in user_path.glob("*"):
        try:
            with Image.open(image_path) as img:
                dimensions[img.size] += 1
                modes[img.mode] += 1

        except Exception as error:
            print(f"Error: {image_path.name} - {error}")

    print(f"\n========== {user} ==========")
    print("Image dimensions:", dimensions)
    print("Image modes:", modes)

In [ ]:

from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path(r"E:\Multimodal Biometric Verification System")

SOURCE_PATH = (
    PROJECT_ROOT
    / "Dataset"
    / "Processed"
    / "Face"
    / "Face_Cropped"
)

YOLO_DATASET_PATH = (
    PROJECT_ROOT
    / "Dataset"
    / "YOLO_Experiment"
    / "Face_Classification"
)

USERS = ["User_01", "User_02"]

RANDOM_STATE = 42


In [ ]:

for user in USERS:

    images = sorted((SOURCE_PATH / user).glob("*.png"))

    train_images, temp_images = train_test_split(
        images,
        test_size=0.30,
        random_state=RANDOM_STATE
    )

    val_images, test_images = train_test_split(
        temp_images,
        test_size=0.50,
        random_state=RANDOM_STATE
    )

    splits = {
        "train": train_images,
        "val": val_images,
        "test": test_images
    }

    print(f"\n========== {user} ==========")

    for split_name, split_images in splits.items():
        print(f"{split_name}: {len(split_images)} images")


In [ ]:

if YOLO_DATASET_PATH.exists() and any(YOLO_DATASET_PATH.iterdir()):
    raise FileExistsError(
        f"Destination already contains files: {YOLO_DATASET_PATH}\n"
        "Choose a new empty destination to avoid overwriting data."
    )

for user in USERS:

    images = sorted((SOURCE_PATH / user).glob("*.png"))

    train_images, temp_images = train_test_split(
        images,
        test_size=0.30,
        random_state=RANDOM_STATE
    )

    val_images, test_images = train_test_split(
        temp_images,
        test_size=0.50,
        random_state=RANDOM_STATE
    )

    splits = {
        "train": train_images,
        "val": val_images,
        "test": test_images
    }

    for split_name, split_images in splits.items():

        destination = YOLO_DATASET_PATH / split_name / user
        destination.mkdir(parents=True, exist_ok=True)

        for image_path in split_images:
            shutil.copy2(
                image_path,
                destination / image_path.name
            )

print("YOLO Dataset preparation completed.")


In [ ]:

for split_name in ["train", "val", "test"]:

    print(f"\n========== {split_name.upper()} ==========")

    total = 0

    for user in USERS:

        folder = YOLO_DATASET_PATH / split_name / user
        count = len(list(folder.glob("*.png")))

        total += count

        print(f"{user}: {count}")

    print(f"Total: {total}")


In [ ]:
try:
    import ultralytics
    print("Ultralytics version:", ultralytics.__version__)

except ImportError:
    print("Ultralytics is not installed.")

In [ ]:

from ultralytics import YOLO
from pathlib import Path

# Dataset
DATASET_PATH = YOLO_DATASET_PATH

# Model output
MODEL_OUTPUT_PATH = (
    PROJECT_ROOT
    / "Models"
    / "Face"
    / "YOLO_Experiment"
)

MODEL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Load pretrained YOLO Classification model
model = YOLO("yolo11n-cls.pt")

print("Model loaded successfully.")
print("Dataset:", DATASET_PATH)
print("Output:", MODEL_OUTPUT_PATH)


In [ ]:

train_results = model.train(
    data=str(DATASET_PATH),

    epochs=50,
    imgsz=224,
    batch=16,

    patience=10,

    project=str(MODEL_OUTPUT_PATH),
    name="YOLO11n_Face_Classification",

    seed=42,
    deterministic=True,

    workers=0,

    plots=True,
    verbose=True
)

print("\nTraining completed.")
print("Best model:", model.trainer.best)


In [ ]:
import shutil

BEST_MODEL_PATH = Path(model.trainer.best)

FINAL_MODEL_PATH = (
    MODEL_OUTPUT_PATH / "best.pt"
)

shutil.copy2(
    BEST_MODEL_PATH,
    FINAL_MODEL_PATH
)

print("Best model saved at:")
print(FINAL_MODEL_PATH)

In [ ]:
trained_model = YOLO(str(FINAL_MODEL_PATH))

test_results = trained_model.val(
    data=str(DATASET_PATH),
    split="test",
    imgsz=224,
    batch=16,
    workers=0,
    plots=True
)

print("\nTest Evaluation Completed.")
print(test_results.results_dict)

In [ ]:

from ultralytics import YOLO
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)
from pathlib import Path
import numpy as np

model = YOLO(str(FINAL_MODEL_PATH))

test_path = YOLO_DATASET_PATH / "test"

class_names = ["User_01", "User_02"]

y_true = []
y_pred = []

for class_name in class_names:

    class_path = test_path / class_name

    for image_path in sorted(class_path.glob("*.png")):

        result = model.predict(
            source=str(image_path),
            imgsz=224,
            verbose=False
        )[0]

        predicted_index = int(result.probs.top1)
        predicted_class = result.names[predicted_index]

        y_true.append(class_name)
        y_pred.append(predicted_class)

print("Accuracy:", accuracy_score(y_true, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=class_names,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=class_names))


In [ ]:

from pathlib import Path
from ultralytics import YOLO

UNKNOWN_PATH = Path(
    r"E:\supervised-task2-main\supervised-task2-main\data\raw\others\public_01\face"
)

model = YOLO(str(FINAL_MODEL_PATH))

unknown_images = []

for extension in ["*.png", "*.jpg", "*.jpeg", "*.bmp"]:
    unknown_images.extend(UNKNOWN_PATH.glob(extension))

unknown_images = sorted(unknown_images)

if not unknown_images:
    raise FileNotFoundError(
        f"No images found in: {UNKNOWN_PATH}"
    )

print("Unknown images found:", len(unknown_images))
print("-" * 65)

for image_path in unknown_images:

    result = model.predict(
        source=str(image_path),
        imgsz=224,
        verbose=False
    )[0]

    predicted_class = result.names[result.probs.top1]
    confidence = float(result.probs.top1conf)





    probabilities = result.probs.data.cpu().numpy()

    print(f"Image: {image_path.name}")
    print(f"Prediction: {predicted_class}")
    print(f"Confidence: {confidence:.4f}")

    for index, probability in enumerate(probabilities):
        print(
            f"  {result.names[index]}: {float(probability):.4f}"
        )

    print("-" * 65)


In [ ]:

import pandas as pd
import numpy as np

unknown_results = [
    ("olivetti_01.png", "User_01", 0.9979),
    ("olivetti_02.png", "User_01", 0.9981),
    ("olivetti_03.png", "User_01", 0.9511),
    ("olivetti_04.png", "User_01", 0.9952),
    ("olivetti_05.png", "User_02", 0.7814),
    ("olivetti_06.png", "User_02", 0.6714),
    ("olivetti_07.png", "User_01", 0.7944),
    ("olivetti_08.png", "User_01", 0.9884),
    ("olivetti_09.png", "User_01", 0.9984),
    ("olivetti_10.png", "User_01", 0.5483),
]

df_unknown = pd.DataFrame(
    unknown_results,
    columns=["Image", "Prediction", "Confidence"]
)

print(df_unknown.to_string(index=False))

print("\nConfidence Statistics:")
print(df_unknown["Confidence"].describe())

print("\nPrediction Distribution:")
print(df_unknown["Prediction"].value_counts())


In [ ]:

from pathlib import Path
import pandas as pd
from ultralytics import YOLO

# Paths
KNOWN_TEST_PATH = YOLO_DATASET_PATH / "test"

UNKNOWN_PATH = Path(
    r"E:\supervised-task2-main\supervised-task2-main\data\raw\others\public_01\face"
)

# Load trained model
model = YOLO(str(FINAL_MODEL_PATH))

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp"}

evaluation_data = []

# 1. Evaluate Known Test Images
for class_folder in sorted(KNOWN_TEST_PATH.iterdir()):

    if not class_folder.is_dir():
        continue

    true_identity = class_folder.name

    for image_path in sorted(class_folder.iterdir()):

        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        result = model.predict(
            source=str(image_path),
            imgsz=224,
            verbose=False
        )[0]

        evaluation_data.append({
            "Image": image_path.name,
            "Actual": true_identity,
            "Prediction": result.names[result.probs.top1],
            "Confidence": float(result.probs.top1conf),
            "Group": "Known"
        })


# 2. Evaluate Unknown Images
for image_path in sorted(UNKNOWN_PATH.iterdir()):

    if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue

    result = model.predict(
        source=str(image_path),
        imgsz=224,
        verbose=False
    )[0]

    evaluation_data.append({
        "Image": image_path.name,
        "Actual": "Unknown",
        "Prediction": result.names[result.probs.top1],
        "Confidence": float(result.probs.top1conf),
        "Group": "Unknown"
    })


# 3. Create Evaluation DataFrame
df_evaluation = pd.DataFrame(evaluation_data)

print("Evaluation Samples:")
print(df_evaluation.groupby("Group").size())

print("\nConfidence Statistics:")
print(
    df_evaluation.groupby("Group")["Confidence"]
    .describe()
    .round(4)
)


# 4. Threshold Analysis
thresholds = [0.50, 0.60, 0.70, 0.75, 0.80,
              0.85, 0.90, 0.95, 0.98, 0.99]

known_data = df_evaluation[df_evaluation["Group"] == "Known"]
unknown_data = df_evaluation[df_evaluation["Group"] == "Unknown"]

threshold_results = []

for threshold in thresholds:

    # Known images accepted by the threshold
    known_accepted = (
        known_data["Confidence"] >= threshold
    ).sum()

    # Unknown images rejected by the threshold
    unknown_rejected = (
        unknown_data["Confidence"] < threshold
    ).sum()

    threshold_results.append({
        "Threshold": threshold,
        "Known_Accepted": known_accepted,
        "Known_Acceptance_Rate": (
            known_accepted / len(known_data)
        ),
        "Unknown_Rejected": unknown_rejected,
        "Unknown_Rejection_Rate": (
            unknown_rejected / len(unknown_data)
        )
    })

df_thresholds = pd.DataFrame(threshold_results)

print("\nThreshold Analysis:")
print(df_thresholds.to_string(index=False))


In [ ]:

# STEP 1: Check Webcam, OpenCV and YOLO Model

import cv2
from pathlib import Path
import matplotlib.pyplot as plt

# Project paths
PROJECT_ROOT = Path(r"E:\Multimodal Biometric Verification System")

MODEL_PATH = (
    PROJECT_ROOT / "Models" / "Face" / "YOLO_Experiment" / "best.pt"
)

# Check YOLO model
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"YOLO model not found: {MODEL_PATH}")

print("YOLO Model:", MODEL_PATH)
print("Model file exists:", MODEL_PATH.exists())

# Load Haar Cascade Face Detector
CASCADE_PATH = (
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

face_detector = cv2.CascadeClassifier(CASCADE_PATH)

if face_detector.empty():
    raise RuntimeError("Could not load Haar Cascade Face Detector")

print("Face Detector loaded successfully")

# Open webcam
camera = cv2.VideoCapture(0, cv2.CAP_DSHOW)

if not camera.isOpened():
    camera.release()
    camera = cv2.VideoCapture(0)

if not camera.isOpened():
    raise RuntimeError(
        "Could not open webcam. Check camera permissions or camera index."
    )

# Capture one frame
ret, frame = camera.read()

# Release camera after capturing
camera.release()

if not ret:
    raise RuntimeError("Failed to capture frame from webcam")

# Convert BGR to RGB for display
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

print("Webcam is working!")
print("Frame dimensions:", frame.shape)




In [ ]:

# ============================================================
# YOLO11n LIVE FACE RECOGNITION
# Single-Cell Jupyter Notebook Implementation
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from ultralytics import YOLO
from IPython.display import display, clear_output

# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(
    r"E:\Multimodal Biometric Verification System"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "Models"
    / "Face"
    / "YOLO_Experiment"
    / "best.pt"
)

CAMERA_INDEX = 0
CONFIDENCE_THRESHOLD = 0.80

# ============================================================
# 2. LOAD YOLO MODEL
# ============================================================

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"YOLO model not found: {MODEL_PATH}"
    )

model = YOLO(str(MODEL_PATH))

print("YOLO Model Loaded Successfully")
print("Classes:", model.names)

# ============================================================
# 3. LOAD FACE DETECTOR
# ============================================================

CASCADE_PATH = (
    cv2.data.haarcascades
    + "haarcascade_frontalface_default.xml"
)

face_detector = cv2.CascadeClassifier(CASCADE_PATH)

if face_detector.empty():
    raise RuntimeError(
        "Failed to load Haar Cascade Face Detector"
    )

print("Face Detector Loaded Successfully")

# ============================================================
# 4. OPEN WEBCAM
# ============================================================

camera = cv2.VideoCapture(
    CAMERA_INDEX,
    cv2.CAP_DSHOW
)

if not camera.isOpened():
    camera.release()
    camera = cv2.VideoCapture(CAMERA_INDEX)

if not camera.isOpened():
    raise RuntimeError("Could not open webcam")

camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("Webcam Started")
print("Press Jupyter Interrupt to stop recognition.")

# ============================================================
# 5. LIVE RECOGNITION
# ============================================================

try:

    while True:

        ret, frame = camera.read()

        if not ret:
            print("Failed to capture frame")
            break

        # ----------------------------------------------------
        # Face Detection
        # ----------------------------------------------------

        gray_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )

        detected_faces = face_detector.detectMultiScale(
            gray_frame,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(60, 60)
        )

        # ----------------------------------------------------
        # Process Detected Faces
        # ----------------------------------------------------

        for (x, y, width, height) in detected_faces:

            face_crop = frame[
                y:y + height,
                x:x + width
            ]

            if face_crop.size == 0:
                continue

            # Convert BGR to RGB
            face_crop = cv2.cvtColor(
                face_crop,
                cv2.COLOR_BGR2RGB
            )

            # ------------------------------------------------
            # YOLO Classification
            # ------------------------------------------------

            result = model.predict(
                source=face_crop,
                imgsz=224,
                verbose=False
            )[0]

            # Individual Class Probabilities
            user_01_confidence = float(
                result.probs.data[0]
            )

            user_02_confidence = float(
                result.probs.data[1]
            )

            # Predicted Identity
            predicted_index = result.probs.top1

            predicted_identity = (
                result.names[predicted_index]
            )

            confidence = float(
                result.probs.top1conf
            )

            # ------------------------------------------------
            # Unknown Rejection (Experimental)
            # ------------------------------------------------

            if confidence >= CONFIDENCE_THRESHOLD:

                identity = predicted_identity
                box_color = (0, 200, 0)

            else:

                identity = "Unknown"
                box_color = (0, 0, 255)

            # ------------------------------------------------
            # Draw Bounding Box
            # ------------------------------------------------

            cv2.rectangle(
                frame,
                (x, y),
                (x + width, y + height),
                box_color,
                2
            )

            # Main Prediction Label
            label = f"{identity} | {confidence:.2%}"

            cv2.putText(
                frame,
                label,
                (x, max(y - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                box_color,
                2
            )

            # ------------------------------------------------
            # Display Individual Class Probabilities
            # ------------------------------------------------

            cv2.putText(
                frame,
                f"User_01: {user_01_confidence:.2%}",
                (x, y + height + 25),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 0),
                1
            )

            cv2.putText(
                frame,
                f"User_02: {user_02_confidence:.2%}",
                (x, y + height + 45),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 0),
                1
            )

        # ----------------------------------------------------
        # Display Webcam Frame in Notebook
        # ----------------------------------------------------

        frame_rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        clear_output(wait=True)

        plt.figure(figsize=(9, 6))
        plt.imshow(frame_rgb)
        plt.axis("off")
        plt.title("YOLO11n Live Face Recognition")

        display(plt.gcf())
        plt.close()

except KeyboardInterrupt:

    print("Recognition Stopped")

finally:

    camera.release()
    cv2.destroyAllWindows()

    print("Webcam Released Successfully")
